# ChemBreak V10 Cloud
## Colab Enterprise + Vertex AI

This controller uses the same `ChemBreak_V10_Cloud` GitHub folder as the standard Colab notebook.

Colab Enterprise normally supplies Google Cloud credentials automatically. No personal API key is required.


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import shutil
import google.auth

PROJECT_ID = "rs-foundsecft-mghasemi"
credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
print("Authentication: OK")
print("Detected project:", detected_project)


In [ ]:
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/tmp/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V10_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v10_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"
print(PROJECT_DIR)


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)

RUN_TYPE = "test"
GCS_OUTPUT_URI = ""

RUNTIME_DIR = Path("/tmp/ChemBreak_V10_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

OUTPUT_DIR = Path(f"/tmp/ChemBreak_V10_Results/{RUN_TYPE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    import time
    started = time.time()
    print(f"\n===== V10 {stage.upper()} START =====", flush=True)
    subprocess.run([
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(OUTPUT_DIR),
    ], check=True)
    print(
        f"===== V10 {stage.upper()} DONE | "
        f"elapsed {(time.time()-started)/60:.1f} min =====\n",
        flush=True,
    )


Run stages in this order:

`preflight -> bootstrap -> plan -> inspect -> generate -> validate -> repair -> judge -> refill -> judge -> finalize`

Use Cloud Storage persistence for pilot and production if the runtime may be deleted.
